# ema-second-moment — faded example 2: v Buffer Multi-Step: Checking Convergence to g^2

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-second-moment`. The last cell reports your progress on the `Optimizer: Adam EMA second moment` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA second moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-second-moment`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-second-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA second moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When the gradient is constant across steps, the EMA second-moment buffer converges to `g^2` as `t → ∞`. After `t` steps starting from `v0 = 0`, the exact value is `v_t = g^2 * (1 - beta2^t)`. The bias-correction factor `1 / (1 - beta2^t)` in Adam compensates for the initial under-estimate caused by starting from zero.

## Faded exercise 2

Implement `run_v_buffer(g_const, beta2, n_steps)` that simulates `n_steps` of the Adam second-moment EMA on a scalar gradient `g_const`, starting from `v = 0`. Return a dict with keys `'final_v'` (float), `'expected_v'` (float, the closed-form `g^2 * (1 - beta2**n_steps)`), and `'bias_corrected_v'` (float, `final_v / (1 - beta2**n_steps)`).

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def run_v_buffer(g_const: float, beta2: float, n_steps: int) -> dict:
    v = t.zeros(1)
    g = t.tensor([g_const])
    for _ in range(n_steps):
        v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
    final_v = v.item()
    expected_v = g_const ** 2 * (1 - beta2 ** n_steps)
    bias_corrected_v = final_v / (1 - beta2 ** n_steps)
    return {
        'final_v': final_v,
        'expected_v': expected_v,
        'bias_corrected_v': bias_corrected_v,
    }


def _test():
    import math
    result = run_v_buffer(g_const=2.0, beta2=0.9, n_steps=20)
    # closed-form expected
    g2 = 4.0
    beta2 = 0.9
    expected = g2 * (1 - beta2 ** 20)
    assert abs(result['final_v'] - expected) < 1e-5
    assert abs(result['expected_v'] - expected) < 1e-5
    # bias-corrected should be close to g^2
    assert abs(result['bias_corrected_v'] - g2) < 1e-5


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def run_v_buffer(g_const: float, beta2: float, n_steps: int) -> dict:
    v = t.zeros(1)
    g = t.tensor([g_const])
    for _ in range(n_steps):
        v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
    final_v = v.item()
    expected_v = g_const ** 2 * (1 - beta2 ** n_steps)
    bias_corrected_v = final_v / (1 - beta2 ** n_steps)
    return {
        'final_v': final_v,
        'expected_v': expected_v,
        'bias_corrected_v': bias_corrected_v,
    }
```
</details>